# YOLOv8m Hyperband: Single Year

This notebook runs the single year YOLOv8m Hyperband experiment used for the controlled HPO comparison.

## Experimental protocol

1. Select one dataset year.
2. Build one deterministic 70/15/15 optimization split with seed 42.
3. Run two complete Hyperband cycles with `r=20`, `R=40`, and `eta=2`.
4. Each cycle uses two configurations at 20 epochs, promotes the best one to a fresh 40 epoch run, and also evaluates two new configurations directly at 40 epochs.
5. Rank promotions and final HPO candidates using validation mAP50-95.
6. Freeze the best successful 40 epoch configuration.
7. Retrain the frozen configuration for 40 epochs using seeds 42, 123, and 999.
8. Evaluate each final `best.pt` once on its matching test split.
9. Report mean and sample SD across the three final runs.

## Compute budget

Each Hyperband cycle uses 160 HPO epoch units. Two cycles use exactly 320 HPO epoch units. The three final 40 epoch runs use another 120 epoch units and are reported separately.

Only five parameters are tuned: `lr0`, `lrf`, `momentum`, `weight_decay`, and `scale`. All other training settings remain fixed across candidates.

The notebook is portable for GitHub. It expects the dataset at `./Datasets/Year2021` or `./Datasets/Year2022`, depending on `YEAR`. Local user names and absolute personal paths are not stored in saved experiment tables or manifests.


## Module 1: Imports and experiment configuration

This module defines the dataset year, portable project paths, fixed YOLOv8m settings, the five dimensional search space, Hyperband fidelities, seeds, and the exact 320 epoch HPO budget. Change only `YEAR` to switch between Year2021 and Year2022.


In [ ]:
from pathlib import Path
from datetime import datetime
from itertools import combinations
from statistics import mean, stdev
from typing import Dict, Any, List, Optional, Tuple

import csv
import json
import math
import os
import random
import re
import shutil
import sys
import time
import xml.etree.ElementTree as ET

import numpy as np
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
import torch
import ultralytics
from ultralytics import YOLO


# Dataset selection
YEAR = "2021"
assert YEAR in {"2021", "2022"}
DATASET_TAG = f"Year{YEAR}"

# Portable project layout.
# Expected source dataset location: ./Datasets/Year2021 or ./Datasets/Year2022
WORKSPACE_ROOT = Path.cwd()
DATASETS_ROOT = WORKSPACE_ROOT / "Datasets"
SOURCE_DATASET_ROOT = DATASETS_ROOT / DATASET_TAG

PROJECT_ROOT = WORKSPACE_ROOT / "outputs" / f"yolov8m_hyperband_{DATASET_TAG.lower()}"
RUNS_ROOT = PROJECT_ROOT / "runs"
SPLITS_ROOT = PROJECT_ROOT / "splits"
STATS_ROOT = PROJECT_ROOT / "stats"
YOLO_DATA_ROOT = PROJECT_ROOT / "yolo_data"

# Target classes
CLASS_NAMES = [
    "Waterhemp",
    "Carpetweed",
    "Morningglory",
    "Goosegrass",
    "SpottedSpurge",
    "PalmerAmaranth",
    "Purslane",
    "Ragweed",
]
TARGET_CLASS_SET = set(CLASS_NAMES)

LABEL_ALIASES = {
    "morninglory": "Morningglory",
    "morning glory": "Morningglory",
    "morningglory": "Morningglory",
    "spotted spurge": "SpottedSpurge",
    "spottedspurge": "SpottedSpurge",
    "palmer amaranth": "PalmerAmaranth",
    "palmeramaranth": "PalmerAmaranth",
}

# Split protocol
TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO = 0.15

OPTIMIZATION_SPLIT_SEED = 42
OPTIMIZATION_TRAIN_SEED = 42
FINAL_EVALUATION_SEEDS = [42, 123, 999]

# Shared YOLO runtime settings
MODEL_NAME = "yolov8m.pt"
IMAGE_SIZE = 640
BATCH_SIZE = 16
WORKERS = 8
PATIENCE = 0
IMAGE_PLACEMENT_MODE = "symlink"

# Hyperband fidelity settings
HB_MIN_EPOCHS = 20
HB_MAX_EPOCHS = 40
HB_ETA = 2
HB_CYCLES = 2
HB_RANDOM_SEED = 456

SCREEN_WARMUP_EPOCHS = 1.5
SCREEN_CLOSE_MOSAIC = 5
FULL_WARMUP_EPOCHS = 3.0
FULL_CLOSE_MOSAIC = 10

FIDELITY_CONTROLS = {
    20: {
        "warmup_epochs": SCREEN_WARMUP_EPOCHS,
        "close_mosaic": SCREEN_CLOSE_MOSAIC,
    },
    40: {
        "warmup_epochs": FULL_WARMUP_EPOCHS,
        "close_mosaic": FULL_CLOSE_MOSAIC,
    },
}

FINAL_EPOCHS = 40
FINAL_WARMUP_EPOCHS = 3.0
FINAL_CLOSE_MOSAIC = 10

# Exact Hyperband budget for r=20, R=40, eta=2.
# Per cycle:
# Bracket 1: 2 x 20 + 1 x 40 = 80 epoch units
# Bracket 2: 2 x 40 = 80 epoch units
# Total per cycle = 160 epoch units
HB_HPO_EPOCH_UNITS = HB_CYCLES * 160
EXPECTED_UNIQUE_CONFIGS = HB_CYCLES * 4
EXPECTED_HPO_EVALUATIONS = HB_CYCLES * 5
EXPECTED_FULL_FIDELITY_EVALUATIONS = HB_CYCLES * 3

FINAL_EVALUATION_EPOCH_UNITS = (
    len(FINAL_EVALUATION_SEEDS) * FINAL_EPOCHS
)

if HB_HPO_EPOCH_UNITS != 320:
    raise RuntimeError(
        "Hyperband HPO budget must be exactly 320 epoch units."
    )

if FINAL_EVALUATION_EPOCH_UNITS != 120:
    raise RuntimeError(
        "Final evaluation budget must be 3 x 40 = 120 epoch units."
    )

# Focused five dimensional search space
SEARCH_SPACE_VERSION = "Focused5D_HB20to40_2Cycles_mAP5095_v1"

SEARCH_SPACE = {
    "learning_rate": {
        "low": 0.0010,
        "high": 0.0100,
        "distribution": "log",
    },
    "lrf": {
        "low": 0.0100,
        "high": 0.2000,
        "distribution": "log",
    },
    "momentum": {
        "low": 0.8500,
        "high": 0.9700,
        "distribution": "linear",
    },
    "weight_decay": {
        "low": 0.0001,
        "high": 0.0010,
        "distribution": "log",
    },
    "scale": {
        "low": 0.2000,
        "high": 0.8000,
        "distribution": "linear",
    },
}

SEARCH_PARAMETER_ORDER = [
    "learning_rate",
    "lrf",
    "momentum",
    "weight_decay",
    "scale",
]

# Untuned values are checked only to confirm that the reference baseline lies
# inside the same search domain.
REFERENCE_UNTUNED_VALUES = {
    "learning_rate": 0.01,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "scale": 0.50,
}

for parameter_name, reference_value in REFERENCE_UNTUNED_VALUES.items():
    parameter_spec = SEARCH_SPACE[parameter_name]
    if not (
        float(parameter_spec["low"])
        <= float(reference_value)
        <= float(parameter_spec["high"])
    ):
        raise RuntimeError(
            f"Reference value for {parameter_name}={reference_value} "
            "is outside the declared search range."
        )

# Fixed settings shared by every candidate
FIXED_TRAINING_ARGS = {
    "optimizer": "SGD",
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "nbs": 64,
    "mosaic": 1.0,
    "hsv_h": 0.015,
    "hsv_s": 0.70,
    "hsv_v": 0.40,
    "degrees": 0.0,
    "translate": 0.10,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mixup": 0.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "cache": "disk",
    "cos_lr": False,
    "rect": False,
}

# Strict experiment safeguards
REQUIRE_FULL_HPO_BUDGET = True
REQUIRE_ALL_FINAL_RUNS = True

# Verify that Ultralytics best checkpoint selection aligns with mAP50-95.
STRICT_FITNESS_ALIGNMENT_CHECK = True
FITNESS_ALIGNMENT_TOLERANCE = 0.002

# One timestamp identifies one complete experiment.
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = RUNS_ROOT / f"hb20to40_2cycles_{RUN_TIMESTAMP}"
HPO_TRIALS_ROOT = RUN_ROOT / "hpo"
FINAL_TRAINING_ROOT = RUN_ROOT / "final_training"
TEST_EVAL_ROOT = RUN_ROOT / "test_evaluation"
RUN_STATS_ROOT = RUN_ROOT / "stats"
RUN_PLOTS_ROOT = RUN_ROOT / "plots"

print("Dataset:", DATASET_TAG)
print("Model:", MODEL_NAME)
print("Hyperband:", "r=20, R=40, eta=2, cycles=2")
print("HPO budget:", HB_HPO_EPOCH_UNITS, "epoch units")
print("Final evaluation budget:", FINAL_EVALUATION_EPOCH_UNITS, "epoch units")
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Workers:", WORKERS)
print("Cache:", FIXED_TRAINING_ARGS["cache"])
print("Patience:", PATIENCE)


## Module 2: Annotation parsing

This module reads Pascal VOC XML and common JSON annotation formats, normalizes label aliases, clips boxes to image boundaries, and rejects invalid boxes.


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def normalize_label(raw_label: str, label_aliases: Dict[str, str]) -> str:
    """Normalize one class label using the supplied alias map."""
    cleaned = str(raw_label).strip()
    return label_aliases.get(cleaned.lower(), cleaned)


def read_image_size(image_path: Path) -> Tuple[int, int]:
    """Return image width and height."""
    try:
        with Image.open(image_path) as image:
            return image.size
    except (UnidentifiedImageError, OSError) as error:
        raise ValueError(f"Unreadable image: {image_path}") from error


def parse_pascal_voc_xml(
    xml_path: Path,
    label_aliases: Dict[str, str],
) -> Tuple[Optional[Tuple[int, int]], List[Dict[str, Any]]]:
    """Parse one Pascal VOC XML annotation."""
    root = ET.parse(xml_path).getroot()
    xml_size: Optional[Tuple[int, int]] = None

    size_node = root.find("size")
    if size_node is not None:
        width_text = size_node.findtext("width")
        height_text = size_node.findtext("height")
        if width_text and height_text:
            xml_size = (int(float(width_text)), int(float(height_text)))

    objects: List[Dict[str, Any]] = []

    for object_node in root.findall("object"):
        raw_name = object_node.findtext("name")
        box_node = object_node.find("bndbox")
        if not raw_name or box_node is None:
            continue

        coordinates = [
            box_node.findtext("xmin"),
            box_node.findtext("ymin"),
            box_node.findtext("xmax"),
            box_node.findtext("ymax"),
        ]
        if any(value is None for value in coordinates):
            continue

        try:
            x1, y1, x2, y2 = [float(value) for value in coordinates]
        except (TypeError, ValueError):
            continue

        if x2 <= x1 + 1 or y2 <= y1 + 1:
            continue

        objects.append({
            "label": normalize_label(raw_name, label_aliases),
            "bbox_xyxy": [x1, y1, x2, y2],
        })

    return xml_size, objects


def extract_via_label(
    attributes: Dict[str, Any],
    label_aliases: Dict[str, str],
) -> Optional[str]:
    """Extract one class label from common VIA region attributes."""
    preferred_keys = [
        "label",
        "class",
        "class_name",
        "name",
        "species",
        "type",
        "weed",
        "category",
    ]

    for key in preferred_keys:
        value = attributes.get(key)

        if isinstance(value, str) and value.strip():
            return normalize_label(value, label_aliases)

        if isinstance(value, dict):
            for nested_key, selected in value.items():
                if selected in {True, 1, "1", "true", "True", "yes", "Yes"}:
                    return normalize_label(nested_key, label_aliases)
            if len(value) == 1:
                return normalize_label(next(iter(value.keys())), label_aliases)

    for value in attributes.values():
        if isinstance(value, str) and value.strip():
            return normalize_label(value, label_aliases)

    return None


def parse_json_annotation(
    json_path: Path,
    label_aliases: Dict[str, str],
) -> List[Dict[str, Any]]:
    """Parse LabelMe style or VIA style JSON annotations."""
    data = json.loads(json_path.read_text(encoding="utf-8"))
    objects: List[Dict[str, Any]] = []

    if isinstance(data, dict) and isinstance(data.get("shapes"), list):
        for shape in data["shapes"]:
            label = normalize_label(shape.get("label", ""), label_aliases)
            points = shape.get("points", [])

            if not label or len(points) < 2:
                continue

            try:
                xs = [float(point[0]) for point in points]
                ys = [float(point[1]) for point in points]
            except (TypeError, ValueError, IndexError):
                continue

            x1, x2 = min(xs), max(xs)
            y1, y2 = min(ys), max(ys)

            if x2 > x1 + 1 and y2 > y1 + 1:
                objects.append({
                    "label": label,
                    "bbox_xyxy": [x1, y1, x2, y2],
                })

        return objects

    metadata = data.get("_via_img_metadata", data) if isinstance(data, dict) else {}
    if not isinstance(metadata, dict):
        return objects

    for entry in metadata.values():
        if not isinstance(entry, dict):
            continue

        regions = entry.get("regions", [])
        if isinstance(regions, dict):
            regions = [regions] if "shape_attributes" in regions else list(regions.values())
        if not isinstance(regions, list):
            continue

        for region in regions:
            shape = region.get("shape_attributes", {}) or {}
            attributes = region.get("region_attributes", {}) or {}

            if shape.get("name") != "rect":
                continue

            values = [
                shape.get("x"),
                shape.get("y"),
                shape.get("width"),
                shape.get("height"),
            ]
            if any(value is None for value in values):
                continue

            label = extract_via_label(attributes, label_aliases)
            if not label:
                continue

            try:
                x, y, width, height = [float(value) for value in values]
            except (TypeError, ValueError):
                continue

            if width > 1 and height > 1:
                objects.append({
                    "label": label,
                    "bbox_xyxy": [x, y, x + width, y + height],
                })

    return objects


def clip_box(
    box: List[float],
    image_width: int,
    image_height: int,
) -> Optional[List[float]]:
    """Clip an xyxy box to image boundaries."""
    x1, y1, x2, y2 = box

    x1 = max(0.0, min(float(x1), image_width - 1.0))
    y1 = max(0.0, min(float(y1), image_height - 1.0))
    x2 = max(0.0, min(float(x2), float(image_width)))
    y2 = max(0.0, min(float(y2), float(image_height)))

    if x2 <= x1 + 1 or y2 <= y1 + 1:
        return None

    return [x1, y1, x2, y2]


## Module 3: Dataset scan and audit

This module builds clean records for the selected year, verifies that the eight study classes are present, and creates one class distribution plot when the experiment runs.


In [ ]:
def build_clean_records(
    source_root: Path,
    target_class_set: set,
    label_aliases: Dict[str, str],
) -> Tuple[List[Dict[str, Any]], Dict[str, int]]:
    """Build clean image records from one dataset year."""
    source_root = Path(source_root)

    if not source_root.exists():
        raise FileNotFoundError(f"Source dataset not found: {source_root}")

    image_root = next(
        (
            candidate
            for candidate in [
                source_root / "JPEGImages",
                source_root / "images",
                source_root,
            ]
            if candidate.exists()
        ),
        None,
    )
    if image_root is None:
        raise FileNotFoundError(f"No image folder found under: {source_root}")

    image_paths = sorted(
        path
        for path in image_root.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not image_paths:
        raise FileNotFoundError(f"No images found under: {image_root}")

    image_names = [path.name for path in image_paths]
    if len(image_names) != len(set(image_names)):
        raise ValueError(
            "Duplicate image filenames were found. "
            "Flat YOLO split folders require unique filenames."
        )

    xml_paths = list(source_root.rglob("*.xml"))
    json_paths = list(source_root.rglob("*.json"))

    xml_by_stem = {path.stem: path for path in xml_paths}
    json_by_stem = {path.stem: path for path in json_paths}

    if len(xml_by_stem) != len(xml_paths):
        raise ValueError("Duplicate XML annotation stems were found.")
    if len(json_by_stem) != len(json_paths):
        raise ValueError("Duplicate JSON annotation stems were found.")

    records: List[Dict[str, Any]] = []
    stats = {
        "images_found": len(image_paths),
        "records_kept": 0,
        "missing_annotation": 0,
        "empty_after_filtering": 0,
        "unreadable_images": 0,
        "invalid_boxes_removed": 0,
        "non_target_objects_removed": 0,
        "used_xml": 0,
        "used_json": 0,
    }

    for image_path in image_paths:
        xml_path = xml_by_stem.get(image_path.stem)
        json_path = json_by_stem.get(image_path.stem)

        annotation_size: Optional[Tuple[int, int]] = None
        objects: List[Dict[str, Any]] = []
        source_used: Optional[str] = None

        try:
            if xml_path is not None:
                annotation_size, xml_objects = parse_pascal_voc_xml(
                    xml_path=xml_path,
                    label_aliases=label_aliases,
                )
                if xml_objects:
                    objects = xml_objects
                    source_used = "xml"

            if not objects and json_path is not None:
                json_objects = parse_json_annotation(
                    json_path=json_path,
                    label_aliases=label_aliases,
                )
                if json_objects:
                    objects = json_objects
                    source_used = "json"
                    annotation_size = None

            if not objects:
                stats["missing_annotation"] += 1
                continue

            if annotation_size is None:
                width, height = read_image_size(image_path)
            else:
                width, height = annotation_size

        except Exception as error:
            stats["unreadable_images"] += 1
            print(f"Skipping {image_path.name}: {error}")
            continue

        clean_objects: List[Dict[str, Any]] = []

        for obj in objects:
            label = normalize_label(obj["label"], label_aliases)

            if label not in target_class_set:
                stats["non_target_objects_removed"] += 1
                continue

            clipped_box = clip_box(
                box=obj["bbox_xyxy"],
                image_width=width,
                image_height=height,
            )
            if clipped_box is None:
                stats["invalid_boxes_removed"] += 1
                continue

            clean_objects.append({
                "label": label,
                "bbox_xyxy": clipped_box,
            })

        if not clean_objects:
            stats["empty_after_filtering"] += 1
            continue

        if source_used is None:
            raise RuntimeError(f"Annotation source could not be resolved for {image_path.name}.")

        stats[f"used_{source_used}"] += 1

        records.append({
            "img_path": str(image_path.resolve()),
            "file_name": image_path.name,
            "width": int(width),
            "height": int(height),
            "objects": clean_objects,
        })

    records.sort(key=lambda row: row["file_name"])
    stats["records_kept"] = len(records)

    if not records:
        raise RuntimeError("No usable records remained after annotation filtering.")

    return records, stats


def audit_records(
    records_to_audit: List[Dict[str, Any]],
    class_names: List[str],
) -> Dict[str, Any]:
    """Count boxes and image occurrences for every study class."""
    box_counts = {name: 0 for name in class_names}
    image_counts = {name: 0 for name in class_names}

    for record in records_to_audit:
        labels = [obj["label"] for obj in record["objects"]]

        for label in labels:
            box_counts[label] += 1

        for label in set(labels):
            image_counts[label] += 1

    return {
        "num_images": len(records_to_audit),
        "num_boxes": int(sum(box_counts.values())),
        "box_count_per_class": box_counts,
        "image_count_per_class": image_counts,
        "missing_classes": [
            name for name in class_names if box_counts[name] == 0
        ],
    }


def plot_class_distribution(
    audit: Dict[str, Any],
    class_names: List[str],
    output_root: Path,
    dataset_tag: str,
) -> None:
    """Plot boxes and image occurrences for the eight target classes."""
    output_root.mkdir(parents=True, exist_ok=True)

    positions = np.arange(len(class_names))
    width = 0.38

    plt.figure(figsize=(11, 5))
    plt.bar(
        positions - width / 2,
        [audit["box_count_per_class"][name] for name in class_names],
        width=width,
        label="boxes",
    )
    plt.bar(
        positions + width / 2,
        [audit["image_count_per_class"][name] for name in class_names],
        width=width,
        label="images",
    )
    plt.xticks(positions, class_names, rotation=40, ha="right")
    plt.ylabel("Count")
    plt.title(f"{dataset_tag}: class distribution")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        output_root / "dataset_class_distribution.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()


## Module 4: Deterministic splitting and YOLO export

This module creates reproducible 70/15/15 splits, checks that filenames do not overlap across train, validation, and test sets, verifies class coverage, writes YOLO labels, and stores filename only split manifests for privacy.


In [ ]:
def split_records_deterministically(
    all_records: List[Dict[str, Any]],
    train_ratio: float,
    valid_ratio: float,
    test_ratio: float,
    seed: int,
) -> Dict[str, List[Dict[str, Any]]]:
    """Create one reproducible train, validation, and test split."""
    if abs(train_ratio + valid_ratio + test_ratio - 1.0) > 1e-9:
        raise ValueError("Split ratios must sum to 1.0.")

    if len(all_records) < 3:
        raise ValueError("At least three records are required for a three way split.")

    shuffled = list(all_records)
    random.Random(seed).shuffle(shuffled)

    total = len(shuffled)
    n_train = int(total * train_ratio)
    n_valid = int(total * valid_ratio)

    if n_train < 1 or n_valid < 1 or total - n_train - n_valid < 1:
        raise ValueError("The selected split ratios produce an empty split.")

    return {
        "train": shuffled[:n_train],
        "valid": shuffled[n_train:n_train + n_valid],
        "test": shuffled[n_train + n_valid:],
    }


def verify_no_split_overlap(
    split_records: Dict[str, List[Dict[str, Any]]],
) -> None:
    """Fail if any filename appears in more than one split."""
    filename_sets = {
        split_name: {row["file_name"] for row in rows}
        for split_name, rows in split_records.items()
    }

    for left_name, right_name in combinations(filename_sets.keys(), 2):
        overlap = filename_sets[left_name] & filename_sets[right_name]
        if overlap:
            preview = sorted(overlap)[:10]
            raise RuntimeError(
                f"Data leakage detected between {left_name} and {right_name}: {preview}"
            )


def audit_split_coverage(
    split_records: Dict[str, List[Dict[str, Any]]],
    class_names: List[str],
    fail_on_missing: bool = True,
) -> Dict[str, Any]:
    """Audit class coverage for train, validation, and test splits."""
    report: Dict[str, Any] = {}

    for split_name, rows in split_records.items():
        split_audit = audit_records(
            records_to_audit=rows,
            class_names=class_names,
        )
        report[split_name] = split_audit

        print(
            f"{split_name:8s} "
            f"images={split_audit['num_images']:5d} "
            f"boxes={split_audit['num_boxes']:6d} "
            f"missing={split_audit['missing_classes']}"
        )

        if fail_on_missing and split_audit["missing_classes"]:
            raise RuntimeError(
                f"Split '{split_name}' is missing classes: "
                f"{split_audit['missing_classes']}"
            )

    return report


def remove_file_or_link(path: Path) -> None:
    """Remove one file or symlink."""
    if path.is_symlink() or path.is_file():
        path.unlink()


def clear_split_directory(directory: Path) -> None:
    """Remove stale files from one flat YOLO split directory."""
    directory.mkdir(parents=True, exist_ok=True)

    for path in directory.iterdir():
        if path.is_file() or path.is_symlink():
            path.unlink()
        elif path.is_dir():
            shutil.rmtree(path)


def place_image(
    source: Path,
    destination: Path,
    mode: str,
) -> str:
    """Place an image using symlink, hardlink, or copy fallback."""
    if mode not in {"symlink", "hardlink", "copy"}:
        raise ValueError("mode must be 'symlink', 'hardlink', or 'copy'.")

    source = source.resolve()
    destination.parent.mkdir(parents=True, exist_ok=True)
    remove_file_or_link(destination)

    if mode == "symlink":
        try:
            destination.symlink_to(source)
            return "symlinked"
        except OSError:
            pass

    if mode in {"symlink", "hardlink"}:
        try:
            os.link(source, destination)
            return "hardlinked"
        except OSError:
            pass

    shutil.copy2(source, destination)
    return "copied"


def xyxy_to_yolo(
    box: List[float],
    image_width: int,
    image_height: int,
) -> Tuple[float, float, float, float]:
    """Convert absolute xyxy coordinates to normalized YOLO xywh."""
    x1, y1, x2, y2 = box

    values = (
        ((x1 + x2) / 2.0) / image_width,
        ((y1 + y2) / 2.0) / image_height,
        (x2 - x1) / image_width,
        (y2 - y1) / image_height,
    )

    return tuple(
        max(0.0, min(1.0, float(value)))
        for value in values
    )


def write_yolo_label(
    record: Dict[str, Any],
    output_path: Path,
    class_to_id: Dict[str, int],
) -> None:
    """Write one YOLO detection label file."""
    lines: List[str] = []

    for obj in record["objects"]:
        class_id = class_to_id[obj["label"]]
        xc, yc, box_width, box_height = xyxy_to_yolo(
            box=obj["bbox_xyxy"],
            image_width=record["width"],
            image_height=record["height"],
        )

        lines.append(
            f"{class_id} {xc:.6f} {yc:.6f} {box_width:.6f} {box_height:.6f}"
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def write_dataset_yaml(
    dataset_root: Path,
    class_names: List[str],
) -> Path:
    """Write the temporary Ultralytics dataset YAML file."""
    yaml_path = dataset_root / "dataset.yaml"

    lines = [
        f"path: {dataset_root.resolve().as_posix()}",
        "train: images/train",
        "val: images/valid",
        "test: images/test",
        f"nc: {len(class_names)}",
        "names:",
    ]
    lines.extend(
        f"  {class_id}: {class_name}"
        for class_id, class_name in enumerate(class_names)
    )

    yaml_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return yaml_path


def count_files(
    directory: Path,
    suffix: Optional[str] = None,
) -> int:
    """Count files and symlinks in one directory."""
    paths = [
        path
        for path in directory.iterdir()
        if path.is_file() or path.is_symlink()
    ]

    if suffix is not None:
        paths = [
            path
            for path in paths
            if path.suffix.lower() == suffix.lower()
        ]

    return len(paths)


def check_yolo_dataset_integrity(dataset_root: Path) -> None:
    """Confirm image and label counts for every split."""
    for split_name in ("train", "valid", "test"):
        image_dir = dataset_root / "images" / split_name
        label_dir = dataset_root / "labels" / split_name

        image_count = count_files(image_dir)
        label_count = count_files(label_dir, ".txt")

        print(
            f"{split_name:8s} "
            f"images={image_count:5d} "
            f"labels={label_count:5d}"
        )

        if image_count != label_count:
            raise RuntimeError(
                f"Image and label counts differ in split '{split_name}'."
            )


def prepare_yolo_dataset(
    all_records: List[Dict[str, Any]],
    output_root: Path,
    manifest_root: Path,
    class_names: List[str],
    train_ratio: float,
    valid_ratio: float,
    test_ratio: float,
    image_mode: str,
    split_seed: int,
) -> Tuple[Path, Dict[str, Any]]:
    """Prepare one complete YOLO dataset for a deterministic split seed."""
    output_root = Path(output_root)
    manifest_root = Path(manifest_root)
    class_to_id = {
        class_name: class_id
        for class_id, class_name in enumerate(class_names)
    }

    split_records = split_records_deterministically(
        all_records=all_records,
        train_ratio=train_ratio,
        valid_ratio=valid_ratio,
        test_ratio=test_ratio,
        seed=split_seed,
    )

    verify_no_split_overlap(split_records)

    coverage_report = audit_split_coverage(
        split_records=split_records,
        class_names=class_names,
        fail_on_missing=True,
    )

    image_dirs = {
        split_name: output_root / "images" / split_name
        for split_name in ("train", "valid", "test")
    }
    label_dirs = {
        split_name: output_root / "labels" / split_name
        for split_name in ("train", "valid", "test")
    }

    for directory in [*image_dirs.values(), *label_dirs.values()]:
        clear_split_directory(directory)

    placement_counts = {
        "symlinked": 0,
        "hardlinked": 0,
        "copied": 0,
    }

    manifest = {
        "seed": int(split_seed),
        "ratios": {
            "train": float(train_ratio),
            "valid": float(valid_ratio),
            "test": float(test_ratio),
        },
        "image_mode_requested": image_mode,
        "class_coverage": coverage_report,
        "splits": {},
    }

    manifest_root.mkdir(parents=True, exist_ok=True)

    for split_name, rows in split_records.items():
        filenames: List[str] = []

        for record in rows:
            source = Path(record["img_path"])
            destination = image_dirs[split_name] / record["file_name"]

            placement_action = place_image(
                source=source,
                destination=destination,
                mode=image_mode,
            )
            placement_counts[placement_action] += 1

            label_path = (
                label_dirs[split_name]
                / f"{Path(record['file_name']).stem}.txt"
            )
            write_yolo_label(
                record=record,
                output_path=label_path,
                class_to_id=class_to_id,
            )

            filenames.append(record["file_name"])

        (manifest_root / f"{split_name}_filenames.txt").write_text(
            "\n".join(filenames) + "\n",
            encoding="utf-8",
        )

        manifest["splits"][split_name] = {
            "count": len(rows),
            "filenames": filenames,
        }

    manifest["actual_placement_counts"] = placement_counts

    (manifest_root / "split_manifest.json").write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )

    dataset_yaml = write_dataset_yaml(
        dataset_root=output_root,
        class_names=class_names,
    )

    check_yolo_dataset_integrity(output_root)

    return dataset_yaml, coverage_report


## Module 5: Metrics and one training run

This module contains the shared YOLOv8m training helper. Every required setting is passed explicitly, each candidate starts from the same pretrained initialization, and validation mAP50-95 is extracted from the frozen `best.pt`.


In [ ]:
def extract_box_metrics(
    metrics: Any,
    expected_class_names: List[str],
) -> Dict[str, Any]:
    """Extract standard detection metrics and per class mAP50-95."""
    box = getattr(metrics, "box", None)

    if box is None:
        raise RuntimeError("Ultralytics returned no box detection metrics.")

    precision = float(box.mp)
    recall = float(box.mr)
    map50 = float(box.map50)
    map50_95 = float(box.map)

    names_object = getattr(metrics, "names", {})
    if isinstance(names_object, dict):
        class_name_lookup = {
            int(class_id): str(class_name)
            for class_id, class_name in names_object.items()
        }
    else:
        class_name_lookup = {
            class_id: str(class_name)
            for class_id, class_name in enumerate(names_object)
        }

    per_class_rows: List[Dict[str, Any]] = []

    if hasattr(box, "ap") and hasattr(box, "ap_class_index"):
        ap_values = np.asarray(box.ap, dtype=float).reshape(-1)
        class_ids = np.asarray(box.ap_class_index, dtype=int).reshape(-1)

        for class_id, ap_value in zip(
            class_ids.tolist(),
            ap_values.tolist(),
        ):
            per_class_rows.append({
                "class_id": int(class_id),
                "class_name": class_name_lookup.get(
                    int(class_id),
                    str(class_id),
                ),
                "map50_95": float(ap_value),
            })
    else:
        map_values = [float(value) for value in box.maps]

        for class_id, ap_value in enumerate(map_values):
            per_class_rows.append({
                "class_id": int(class_id),
                "class_name": class_name_lookup.get(
                    int(class_id),
                    str(class_id),
                ),
                "map50_95": float(ap_value),
            })

    represented_ids = {
        int(row["class_id"])
        for row in per_class_rows
    }
    expected_ids = set(range(len(expected_class_names)))
    missing_ids = sorted(expected_ids - represented_ids)

    if missing_ids:
        missing_names = [
            expected_class_names[class_id]
            for class_id in missing_ids
        ]
        raise RuntimeError(
            "Metrics are missing expected classes: "
            f"{missing_names}. Check split coverage."
        )

    sorted_ap_values = sorted(
        float(row["map50_95"])
        for row in per_class_rows
    )
    bottom_count = min(3, len(sorted_ap_values))
    bottom3_mean = float(mean(sorted_ap_values[:bottom_count]))

    return {
        "precision": precision,
        "recall": recall,
        "map50": map50,
        "map50_95": map50_95,
        "fitness": map50_95,
        "bottom3_mean_map50_95": bottom3_mean,
        "per_class_map50_95": per_class_rows,
    }


def read_epoch_map50_95_values(
    results_csv: Path,
) -> Tuple[str, List[float]]:
    """Read epoch level validation mAP50-95 values."""
    results_csv = Path(results_csv)

    if not results_csv.exists():
        raise FileNotFoundError(f"Training results.csv not found: {results_csv}")

    with results_csv.open(
        "r",
        newline="",
        encoding="utf-8-sig",
    ) as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        fieldnames = reader.fieldnames or []

    candidate_columns = [
        column_name
        for column_name in fieldnames
        if "mAP50-95" in column_name
    ]

    if not candidate_columns:
        raise RuntimeError(
            "No mAP50-95 column was found in results.csv. "
            f"Available columns: {fieldnames}"
        )

    metric_column = next(
        (
            column_name
            for column_name in candidate_columns
            if "metrics/" in column_name
        ),
        candidate_columns[0],
    )

    values: List[float] = []

    for row in rows:
        raw_value = row.get(metric_column, "")

        try:
            value = float(raw_value)
        except (TypeError, ValueError):
            continue

        if math.isfinite(value):
            values.append(value)

    if not values:
        raise RuntimeError(
            f"No numeric values were found in column: {metric_column}"
        )

    return metric_column, values


def verify_ultralytics_fitness_alignment(
    results_csv: Path,
    trainer_best_fitness: Optional[float],
    tolerance: float,
    strict: bool,
) -> Dict[str, Any]:
    """Check trainer best fitness against the maximum epoch mAP50-95."""
    metric_column, map95_values = read_epoch_map50_95_values(results_csv)
    max_epoch_map95 = float(max(map95_values))

    if trainer_best_fitness is None:
        report = {
            "status": "UNVERIFIED",
            "metric_column": metric_column,
            "max_epoch_map50_95": max_epoch_map95,
            "trainer_best_fitness": None,
            "absolute_difference": None,
            "tolerance": float(tolerance),
        }

        if strict:
            raise RuntimeError(
                "trainer.best_fitness was unavailable, so checkpoint "
                "fitness alignment could not be verified."
            )

        return report

    trainer_best_fitness = float(trainer_best_fitness)

    if not math.isfinite(trainer_best_fitness):
        raise RuntimeError("trainer.best_fitness is not finite.")

    difference = abs(trainer_best_fitness - max_epoch_map95)
    status = "PASS" if difference <= tolerance else "FAIL"

    report = {
        "status": status,
        "metric_column": metric_column,
        "max_epoch_map50_95": max_epoch_map95,
        "trainer_best_fitness": trainer_best_fitness,
        "absolute_difference": float(difference),
        "tolerance": float(tolerance),
    }

    if strict and status != "PASS":
        raise RuntimeError(
            "Checkpoint fitness does not align with mAP50-95. "
            f"best_fitness={trainer_best_fitness:.6f}, "
            f"max_epoch_map50_95={max_epoch_map95:.6f}, "
            f"difference={difference:.6f}."
        )

    return report


def train_and_validate_configuration(
    *,
    learning_rate: float,
    lrf: float,
    momentum: float,
    weight_decay: float,
    scale: float,
    dataset_yaml: Path,
    project_dir: Path,
    run_name: str,
    model_name: str,
    epochs: int,
    image_size: int,
    batch_size: int,
    patience: int,
    workers: int,
    seed: int,
    warmup_epochs: float,
    close_mosaic: int,
    fixed_training_args: Dict[str, Any],
    expected_class_names: List[str],
    strict_fitness_alignment_check: bool,
    fitness_alignment_tolerance: float,
    device: Optional[str] = None,
) -> Dict[str, Any]:
    """Train one configuration from fresh pretrained weights and validate best.pt."""
    dataset_yaml = Path(dataset_yaml)
    project_dir = Path(project_dir)

    if not dataset_yaml.exists():
        raise FileNotFoundError(f"Dataset YAML not found: {dataset_yaml}")
    if learning_rate <= 0 or lrf <= 0:
        raise ValueError("learning_rate and lrf must be greater than 0.")
    if not 0.0 < momentum < 1.0:
        raise ValueError("momentum must be between 0 and 1.")
    if weight_decay < 0:
        raise ValueError("weight_decay cannot be negative.")
    if not 0.0 <= scale <= 1.0:
        raise ValueError("scale must be between 0 and 1.")
    if epochs < 1:
        raise ValueError("epochs must be at least 1.")
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1.")
    if workers < 0:
        raise ValueError("workers cannot be negative.")
    if warmup_epochs < 0:
        raise ValueError("warmup_epochs cannot be negative.")
    if close_mosaic < 0:
        raise ValueError("close_mosaic cannot be negative.")

    if device is None:
        device = "0" if torch.cuda.is_available() else "cpu"

    using_cuda = str(device) != "cpu" and torch.cuda.is_available()

    if using_cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    project_dir.mkdir(parents=True, exist_ok=True)

    train_kwargs = dict(fixed_training_args)
    train_kwargs.update({
        "data": str(dataset_yaml.resolve()),
        "project": str(project_dir.resolve()),
        "name": run_name,
        "epochs": int(epochs),
        "imgsz": int(image_size),
        "batch": int(batch_size),
        "patience": int(patience),
        "workers": int(workers),
        "device": device,
        "seed": int(seed),
        "deterministic": True,
        "pretrained": True,
        "val": True,
        "save": True,
        "save_period": -1,
        "plots": False,
        "verbose": True,
        "amp": bool(using_cuda),
        "lr0": float(learning_rate),
        "lrf": float(lrf),
        "momentum": float(momentum),
        "weight_decay": float(weight_decay),
        "scale": float(scale),
        "warmup_epochs": float(warmup_epochs),
        "close_mosaic": int(close_mosaic),
    })

    print("\nStarting:", run_name)
    print(
        json.dumps(
            {
                "learning_rate": float(learning_rate),
                "lrf": float(lrf),
                "momentum": float(momentum),
                "weight_decay": float(weight_decay),
                "scale": float(scale),
                "epochs": int(epochs),
                "seed": int(seed),
            },
            indent=2,
        )
    )

    model = YOLO(model_name)

    train_start = time.perf_counter()
    train_result = model.train(**train_kwargs)
    training_seconds = time.perf_counter() - train_start

    train_save_dir = getattr(train_result, "save_dir", None)

    if train_save_dir is None:
        trainer = getattr(model, "trainer", None)
        if trainer is not None:
            train_save_dir = getattr(trainer, "save_dir", None)

    if train_save_dir is None:
        raise RuntimeError("Training completed but save_dir could not be resolved.")

    train_save_dir = Path(train_save_dir)
    best_checkpoint = train_save_dir / "weights" / "best.pt"
    results_csv = train_save_dir / "results.csv"

    if not best_checkpoint.exists():
        raise FileNotFoundError(f"best.pt not found: {best_checkpoint}")
    if not results_csv.exists():
        raise FileNotFoundError(f"results.csv not found: {results_csv}")

    trainer_best_fitness: Optional[float] = None
    trainer = getattr(model, "trainer", None)

    if trainer is not None:
        raw_best_fitness = getattr(trainer, "best_fitness", None)
        if raw_best_fitness is not None:
            trainer_best_fitness = float(raw_best_fitness)

    alignment_report = verify_ultralytics_fitness_alignment(
        results_csv=results_csv,
        trainer_best_fitness=trainer_best_fitness,
        tolerance=fitness_alignment_tolerance,
        strict=strict_fitness_alignment_check,
    )

    best_model = YOLO(str(best_checkpoint))

    validation_start = time.perf_counter()
    validation_metrics = best_model.val(
        data=str(dataset_yaml.resolve()),
        split="val",
        imgsz=int(image_size),
        batch=int(batch_size),
        workers=int(workers),
        device=device,
        project=str(project_dir.resolve()),
        name=f"{run_name}_validation",
        plots=False,
        save_json=False,
        verbose=False,
    )
    validation_seconds = time.perf_counter() - validation_start

    metric_data = extract_box_metrics(
        metrics=validation_metrics,
        expected_class_names=expected_class_names,
    )

    peak_gpu_memory_gb: Optional[float] = None

    if using_cuda:
        peak_gpu_memory_gb = (
            torch.cuda.max_memory_allocated() / (1024 ** 3)
        )

    print(
        f"Completed {run_name}: "
        f"mAP50-95={metric_data['map50_95']:.6f}, "
        f"P={metric_data['precision']:.6f}, "
        f"R={metric_data['recall']:.6f}"
    )

    return {
        "status": "PASS",
        "run_name": run_name,
        "metrics": metric_data,
        "fitness_alignment": alignment_report,
        "timing": {
            "training_seconds": float(training_seconds),
            "validation_seconds": float(validation_seconds),
            "total_seconds": float(training_seconds + validation_seconds),
        },
        "hardware": {
            "device": str(device),
            "gpu_name": (
                torch.cuda.get_device_name(0)
                if using_cuda
                else None
            ),
            "peak_gpu_memory_gb": (
                float(peak_gpu_memory_gb)
                if peak_gpu_memory_gb is not None
                else None
            ),
        },
        "paths": {
            "training_folder": train_save_dir,
            "best_checkpoint": best_checkpoint,
            "results_csv": results_csv,
        },
    }


## Module 6: Hyperband plan, ranking, and summaries

This module builds the deterministic two cycle Hyperband plan, audits the exact 320 epoch HPO budget, samples the five tuned parameters, ranks candidates by validation mAP50-95, and prepares summary tables.


In [ ]:
def safe_float(value: Any) -> Optional[float]:
    """Convert a value to a finite float."""
    try:
        converted = float(value)
    except (TypeError, ValueError):
        return None

    return converted if math.isfinite(converted) else None


def save_rows_to_csv_safe(
    rows: List[Dict[str, Any]],
    output_csv: Path,
) -> None:
    """Save a list of dictionaries to CSV."""
    if not rows:
        return

    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    fieldnames: List[str] = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)

    try:
        with output_csv.open(
            "w",
            newline="",
            encoding="utf-8",
        ) as file:
            writer = csv.DictWriter(
                file,
                fieldnames=fieldnames,
            )
            writer.writeheader()
            writer.writerows(rows)

    except PermissionError:
        backup_path = output_csv.with_name(
            f"{output_csv.stem}_backup_"
            f"{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            f"{output_csv.suffix}"
        )

        with backup_path.open(
            "w",
            newline="",
            encoding="utf-8",
        ) as file:
            writer = csv.DictWriter(
                file,
                fieldnames=fieldnames,
            )
            writer.writeheader()
            writer.writerows(rows)

        print(
            "CSV was open elsewhere. Backup written to:",
            backup_path.name,
        )


def save_json(
    data: Dict[str, Any],
    output_json: Path,
) -> None:
    """Save one dictionary as JSON."""
    output_json = Path(output_json)
    output_json.parent.mkdir(parents=True, exist_ok=True)
    output_json.write_text(
        json.dumps(data, indent=2),
        encoding="utf-8",
    )


def privacy_safe_text(
    value: Any,
    workspace_root: Path,
) -> str:
    """Redact likely personal home paths from persisted error text."""
    text = str(value)

    try:
        workspace_text = str(workspace_root.resolve())
        text = text.replace(workspace_text, "<WORKSPACE>")
    except OSError:
        pass

    text = re.sub(
        r"[A-Za-z]:\\Users\\[^\\\s]+",
        "<USER_HOME>",
        text,
    )
    text = re.sub(
        r"/" + "home" + r"/[^/\s]+",
        "<USER_HOME>",
        text,
    )
    text = re.sub(
        r"/" + "Users" + r"/[^/\s]+",
        "<USER_HOME>",
        text,
    )
    return text


def relative_output_path(
    path: Path,
    workspace_root: Path,
) -> str:
    """Return a portable path for manifests and console summaries."""
    path = Path(path)

    try:
        return path.resolve().relative_to(
            workspace_root.resolve()
        ).as_posix()
    except (ValueError, OSError):
        return path.name


def sample_parameter(
    rng: random.Random,
    spec: Dict[str, Any],
) -> float:
    """Sample one parameter using its declared search geometry."""
    low = float(spec["low"])
    high = float(spec["high"])
    distribution = str(spec["distribution"]).lower()

    if not low < high:
        raise ValueError(
            "Search space lower bound must be smaller than upper bound."
        )

    if distribution == "log":
        if low <= 0:
            raise ValueError(
                "Log uniform lower bounds must be greater than 0."
            )
        return float(
            math.exp(
                rng.uniform(
                    math.log(low),
                    math.log(high),
                )
            )
        )

    if distribution == "linear":
        return float(rng.uniform(low, high))

    raise ValueError(
        f"Unsupported search distribution: {distribution}"
    )


def sample_hyperband_config(
    rng: random.Random,
    config_id: str,
    search_space: Dict[str, Dict[str, Any]],
    parameter_order: List[str],
) -> Dict[str, Any]:
    """Sample one five dimensional Hyperband configuration."""
    config: Dict[str, Any] = {
        "config_id": str(config_id)
    }

    for parameter_name in parameter_order:
        config[parameter_name] = sample_parameter(
            rng=rng,
            spec=search_space[parameter_name],
        )

    return config


def config_signature(
    config: Dict[str, Any],
    parameter_order: List[str],
) -> Tuple[float, ...]:
    """Return a stable numeric signature for duplicate detection."""
    return tuple(
        round(float(config[parameter_name]), 15)
        for parameter_name in parameter_order
    )


def get_fidelity_controls(
    budget_epochs: int,
    fidelity_controls: Dict[int, Dict[str, Any]],
) -> Tuple[float, int]:
    """Return the warmup and close mosaic settings for one fidelity."""
    budget_epochs = int(budget_epochs)

    if budget_epochs not in fidelity_controls:
        raise ValueError(
            f"No fidelity controls are defined for {budget_epochs} epochs."
        )

    controls = fidelity_controls[budget_epochs]
    return (
        float(controls["warmup_epochs"]),
        int(controls["close_mosaic"]),
    )


def build_hyperband_plan(
    min_epochs: int,
    max_epochs: int,
    eta: int,
    n_cycles: int,
    random_seed: int,
    search_space: Dict[str, Dict[str, Any]],
    parameter_order: List[str],
    expected_unique_configs: int,
    expected_hpo_evaluations: int,
    expected_full_fidelity_evaluations: int,
    expected_hpo_epoch_units: int,
) -> List[Dict[str, Any]]:
    """Build and audit the complete Hyperband plan."""
    min_epochs = int(min_epochs)
    max_epochs = int(max_epochs)
    eta = int(eta)
    n_cycles = int(n_cycles)

    if min_epochs != 20 or max_epochs != 40:
        raise ValueError(
            "This experiment is fixed to 20 and 40 epoch fidelities."
        )
    if eta != 2:
        raise ValueError(
            "This experiment is fixed to eta=2."
        )
    if n_cycles < 1:
        raise ValueError(
            "n_cycles must be at least 1."
        )

    s_max = int(
        math.floor(
            math.log(max_epochs / min_epochs, eta)
        )
    )

    if s_max != 1:
        raise RuntimeError(
            f"Unexpected Hyperband s_max={s_max}; expected 1."
        )

    rng = random.Random(int(random_seed))
    brackets: List[Dict[str, Any]] = []
    seen_signatures = set()
    config_counter = 1

    for cycle_index in range(1, n_cycles + 1):
        cycle_id = f"HB{cycle_index}"

        for bracket_number, s in enumerate(
            reversed(range(s_max + 1)),
            start=1,
        ):
            initial_n = int(
                math.ceil(
                    ((s_max + 1) / (s + 1))
                    * (eta ** s)
                )
            )
            initial_budget = max_epochs / (eta ** s)

            budgets: List[int] = []
            for stage_index in range(s + 1):
                budget = int(
                    round(
                        initial_budget
                        * (eta ** stage_index)
                    )
                )
                budget = max(
                    min_epochs,
                    min(max_epochs, budget),
                )
                if budget not in budgets:
                    budgets.append(budget)

            configs: List[Dict[str, Any]] = []
            for _ in range(initial_n):
                config_id = f"C{config_counter:03d}"

                config = sample_hyperband_config(
                    rng=rng,
                    config_id=config_id,
                    search_space=search_space,
                    parameter_order=parameter_order,
                )

                signature = config_signature(
                    config=config,
                    parameter_order=parameter_order,
                )

                if signature in seen_signatures:
                    raise RuntimeError(
                        "Hyperband sampled a duplicate configuration."
                    )

                seen_signatures.add(signature)
                configs.append(config)
                config_counter += 1

            brackets.append({
                "cycle_index": int(cycle_index),
                "cycle_id": cycle_id,
                "bracket_id": (
                    f"{cycle_id}_B{bracket_number}"
                ),
                "bracket_number": int(bracket_number),
                "s": int(s),
                "initial_n": int(initial_n),
                "budgets": budgets,
                "configs": configs,
            })

    unique_config_count = sum(
        len(bracket["configs"])
        for bracket in brackets
    )

    if unique_config_count != int(expected_unique_configs):
        raise RuntimeError(
            "Hyperband unique configuration count does not match "
            f"the expected value: {unique_config_count} != "
            f"{expected_unique_configs}."
        )

    planned_epoch_units = 0
    planned_evaluations = 0
    planned_full_fidelity_evaluations = 0

    for bracket in brackets:
        active_count = int(bracket["initial_n"])

        for stage_index, budget in enumerate(
            bracket["budgets"]
        ):
            planned_epoch_units += (
                active_count * int(budget)
            )
            planned_evaluations += active_count

            if int(budget) == max_epochs:
                planned_full_fidelity_evaluations += active_count

            if stage_index < len(bracket["budgets"]) - 1:
                active_count = max(
                    1,
                    int(math.floor(active_count / eta)),
                )

    if planned_epoch_units != int(expected_hpo_epoch_units):
        raise RuntimeError(
            "Hyperband planned epoch units do not match the "
            f"declared budget: {planned_epoch_units} != "
            f"{expected_hpo_epoch_units}."
        )

    if planned_evaluations != int(expected_hpo_evaluations):
        raise RuntimeError(
            "Hyperband evaluation count does not match the "
            f"expected value: {planned_evaluations} != "
            f"{expected_hpo_evaluations}."
        )

    if (
        planned_full_fidelity_evaluations
        != int(expected_full_fidelity_evaluations)
    ):
        raise RuntimeError(
            "Hyperband full fidelity evaluation count does not "
            "match the expected value."
        )

    return brackets


def flatten_hyperband_plan(
    brackets: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Flatten initial Hyperband assignments into CSV rows."""
    rows: List[Dict[str, Any]] = []

    for bracket in brackets:
        for config in bracket["configs"]:
            rows.append({
                "cycle_index": bracket["cycle_index"],
                "cycle_id": bracket["cycle_id"],
                "bracket_id": bracket["bracket_id"],
                "bracket_number": bracket["bracket_number"],
                "s": bracket["s"],
                "initial_n": bracket["initial_n"],
                "budgets": json.dumps(
                    bracket["budgets"]
                ),
                **config,
            })

    return rows


def rank_hyperband_rows(
    rows: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Rank successful rows by mAP50-95 with deterministic tie breaks."""
    eligible: List[Dict[str, Any]] = []

    for row in rows:
        if str(row.get("status", "")).upper() != "PASS":
            continue

        map95 = safe_float(row.get("val_map50_95"))
        map50 = safe_float(row.get("val_map50"))
        recall = safe_float(row.get("val_recall"))

        if map95 is None or map50 is None or recall is None:
            continue

        eligible.append(row)

    return sorted(
        eligible,
        key=lambda row: (
            -float(row["val_map50_95"]),
            -float(row["val_map50"]),
            -float(row["val_recall"]),
            str(row["config_id"]),
        ),
    )


def summarize_metric_mean_sd(
    rows: List[Dict[str, Any]],
    metric_names: List[str],
) -> List[Dict[str, Any]]:
    """Return mean, sample SD, minimum, and maximum for each metric."""
    summary: List[Dict[str, Any]] = []

    for metric_name in metric_names:
        values = [
            value
            for value in (
                safe_float(row.get(metric_name))
                for row in rows
            )
            if value is not None
        ]

        if not values:
            summary.append({
                "metric": metric_name,
                "n": 0,
                "mean": "",
                "sd": "",
                "min": "",
                "max": "",
            })
        elif len(values) == 1:
            summary.append({
                "metric": metric_name,
                "n": 1,
                "mean": values[0],
                "sd": "",
                "min": values[0],
                "max": values[0],
            })
        else:
            summary.append({
                "metric": metric_name,
                "n": len(values),
                "mean": mean(values),
                "sd": stdev(values),
                "min": min(values),
                "max": max(values),
            })

    return summary


def summarize_per_class_mean_sd(
    per_class_rows: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Summarize per class test mAP50-95 across final seeds."""
    if not per_class_rows:
        return []

    summary: List[Dict[str, Any]] = []

    class_names = sorted(
        {row["class_name"] for row in per_class_rows}
    )

    for class_name in class_names:
        values = [
            float(row["test_map50_95"])
            for row in per_class_rows
            if row["class_name"] == class_name
        ]

        summary.append({
            "class_name": class_name,
            "n": len(values),
            "mean_test_map50_95": mean(values),
            "sd_test_map50_95": (
                ""
                if len(values) == 1
                else stdev(values)
            ),
            "min_test_map50_95": min(values),
            "max_test_map50_95": max(values),
        })

    return summary


def boundary_proximity_report(
    selected_config: Dict[str, Any],
    search_space: Dict[str, Dict[str, Any]],
    parameter_order: List[str],
    boundary_fraction: float = 0.10,
) -> List[Dict[str, Any]]:
    """Report the selected value position inside each search range."""
    report: List[Dict[str, Any]] = []

    for parameter_name in parameter_order:
        low = float(search_space[parameter_name]["low"])
        high = float(search_space[parameter_name]["high"])
        value = float(selected_config[parameter_name])

        if search_space[parameter_name]["distribution"] == "log":
            low_t = math.log(low)
            high_t = math.log(high)
            value_t = math.log(value)
        else:
            low_t = low
            high_t = high
            value_t = value

        normalized_position = (
            (value_t - low_t)
            / (high_t - low_t)
        )

        if normalized_position <= boundary_fraction:
            boundary_flag = "LOW"
        elif normalized_position >= 1.0 - boundary_fraction:
            boundary_flag = "HIGH"
        else:
            boundary_flag = "INTERIOR"

        report.append({
            "parameter": parameter_name,
            "value": value,
            "low": low,
            "high": high,
            "normalized_position": normalized_position,
            "boundary_flag": boundary_flag,
        })

    return report


## Module 7: Compact experiment plots

This module keeps the visual output focused. It plots the Hyperband validation process and the final mean plus sample SD for the main test metrics. Together with the dataset class distribution, the notebook produces three custom figures.


In [ ]:
def plot_hyperband_progress(
    rows: List[Dict[str, Any]],
    output_root: Path,
    dataset_tag: str,
) -> None:
    """Plot validation mAP50-95 by HPO evaluation and fidelity."""
    successful_rows = sorted(
        [
            row
            for row in rows
            if row.get("status") == "PASS"
        ],
        key=lambda row: int(row["evaluation_index"]),
    )

    if not successful_rows:
        return

    output_root.mkdir(parents=True, exist_ok=True)

    screen_rows = [
        row
        for row in successful_rows
        if int(row["budget_epochs"]) == 20
    ]
    full_rows = [
        row
        for row in successful_rows
        if int(row["budget_epochs"]) == 40
    ]

    plt.figure(figsize=(9, 5))

    if screen_rows:
        plt.scatter(
            [int(row["evaluation_index"]) for row in screen_rows],
            [float(row["val_map50_95"]) for row in screen_rows],
            marker="o",
            label="20 epoch evaluation",
        )

    if full_rows:
        plt.scatter(
            [int(row["evaluation_index"]) for row in full_rows],
            [float(row["val_map50_95"]) for row in full_rows],
            marker="s",
            label="40 epoch evaluation",
        )

        running_best_x: List[int] = []
        running_best_y: List[float] = []
        current_best = -float("inf")

        for row in full_rows:
            current_best = max(
                current_best,
                float(row["val_map50_95"]),
            )
            running_best_x.append(
                int(row["evaluation_index"])
            )
            running_best_y.append(current_best)

        plt.plot(
            running_best_x,
            running_best_y,
            linestyle="--",
            label="running best at 40 epochs",
        )

    plt.xlabel("Hyperband evaluation")
    plt.ylabel("Validation mAP50-95")
    plt.title(f"{dataset_tag}: Hyperband validation progress")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        output_root / "hyperband_validation_progress.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()


def plot_final_test_metrics(
    final_rows: List[Dict[str, Any]],
    output_root: Path,
    dataset_tag: str,
) -> None:
    """Plot mean and sample SD for the main final test metrics."""
    successful_rows = [
        row
        for row in final_rows
        if row["status"] == "PASS"
    ]

    if not successful_rows:
        return

    metric_keys = [
        "test_precision",
        "test_recall",
        "test_map50",
        "test_map50_95",
    ]
    labels = [
        "Precision",
        "Recall",
        "mAP50",
        "mAP50-95",
    ]

    metric_means: List[float] = []
    metric_sds: List[float] = []

    for metric_key in metric_keys:
        values = [
            float(row[metric_key])
            for row in successful_rows
        ]
        metric_means.append(mean(values))
        metric_sds.append(
            stdev(values)
            if len(values) > 1
            else 0.0
        )

    output_root.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(8, 5))
    plt.bar(
        labels,
        metric_means,
        yerr=metric_sds,
        capsize=4,
    )
    plt.ylabel("Score")
    plt.ylim(0.0, 1.0)
    plt.title(
        f"{dataset_tag}: final test mean and sample SD"
    )
    plt.tight_layout()
    plt.savefig(
        output_root / "final_test_metrics_mean_sd.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()


## Module 8: Test evaluation

This helper evaluates one frozen final checkpoint on the test split. It is not called during Hyperband promotion or HPO selection.


In [ ]:
def evaluate_checkpoint_on_test_split(
    *,
    checkpoint_path: Path,
    dataset_yaml: Path,
    output_root: Path,
    run_name: str,
    expected_class_names: List[str],
    image_size: int,
    batch_size: int,
    workers: int,
    device: Optional[str] = None,
) -> Dict[str, Any]:
    """Evaluate one frozen checkpoint on its test split."""
    checkpoint_path = Path(checkpoint_path)
    dataset_yaml = Path(dataset_yaml)
    output_root = Path(output_root)

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {checkpoint_path}"
        )

    if not dataset_yaml.exists():
        raise FileNotFoundError(
            f"Dataset YAML not found: {dataset_yaml}"
        )

    if device is None:
        device = "0" if torch.cuda.is_available() else "cpu"

    output_root.mkdir(parents=True, exist_ok=True)

    model = YOLO(str(checkpoint_path))

    test_start = time.perf_counter()

    test_metrics = model.val(
        data=str(dataset_yaml.resolve()),
        split="test",
        imgsz=int(image_size),
        batch=int(batch_size),
        workers=int(workers),
        device=device,
        project=str(output_root.resolve()),
        name=run_name,
        plots=False,
        save_json=False,
        verbose=False,
    )

    test_seconds = time.perf_counter() - test_start

    metric_data = extract_box_metrics(
        metrics=test_metrics,
        expected_class_names=expected_class_names,
    )

    per_class_rows = [
        {
            "class_id": row["class_id"],
            "class_name": row["class_name"],
            "test_map50_95": row["map50_95"],
        }
        for row in metric_data["per_class_map50_95"]
    ]

    return {
        "test_precision": metric_data["precision"],
        "test_recall": metric_data["recall"],
        "test_map50": metric_data["map50"],
        "test_map50_95": metric_data["map50_95"],
        "test_bottom3_mean_map50_95": (
            metric_data["bottom3_mean_map50_95"]
        ),
        "test_seconds": float(test_seconds),
        "per_class_rows": per_class_rows,
    }


## Module 9: Complete Hyperband workflow

This module runs the complete experiment from dataset loading through HPO, winner freezing, three seed final evaluation, summaries, plots, and cleanup. It stops rather than reporting a partial HPO budget if any required evaluation fails.


In [ ]:
def create_output_directories() -> None:
    """Create persistent and temporary output directories."""
    for folder in [
        PROJECT_ROOT,
        RUNS_ROOT,
        SPLITS_ROOT,
        STATS_ROOT,
        YOLO_DATA_ROOT,
        RUN_ROOT,
        HPO_TRIALS_ROOT,
        FINAL_TRAINING_ROOT,
        TEST_EVAL_ROOT,
        RUN_STATS_ROOT,
        RUN_PLOTS_ROOT,
    ]:
        folder.mkdir(
            parents=True,
            exist_ok=True,
        )


def run_hyperband_experiment() -> Dict[str, Any]:
    """Run the complete single year Hyperband experiment."""
    if not SOURCE_DATASET_ROOT.exists():
        raise FileNotFoundError(
            "Dataset folder not found. Expected location: "
            f"{DATASETS_ROOT.name}/{DATASET_TAG}"
        )

    create_output_directories()

    workflow_start = time.perf_counter()
    device = (
        "0"
        if torch.cuda.is_available()
        else "cpu"
    )

    plan_csv = RUN_STATS_ROOT / "hyperband_plan.csv"
    hpo_results_csv = RUN_STATS_ROOT / "hyperband_validation_results.csv"
    ranked_full_csv = RUN_STATS_ROOT / "hyperband_full_fidelity_ranked_results.csv"
    selected_json = RUN_STATS_ROOT / "selected_hyperband_config.json"
    boundary_csv = RUN_STATS_ROOT / "selected_config_boundary_report.csv"

    final_csv = RUN_STATS_ROOT / "final_three_seed_test_results.csv"
    final_summary_csv = RUN_STATS_ROOT / "final_three_seed_mean_sd_summary.csv"
    per_class_csv = RUN_STATS_ROOT / "final_test_per_class.csv"
    per_class_summary_csv = RUN_STATS_ROOT / "final_test_per_class_mean_sd.csv"
    manifest_json = RUN_STATS_ROOT / "experiment_manifest.json"

    print("=" * 88)
    print("YOLOv8m HYPERBAND")
    print("=" * 88)
    print("Dataset:", DATASET_TAG)
    print("Model:", MODEL_NAME)
    print("Fitness: validation mAP50-95")
    print("Fidelities:", HB_MIN_EPOCHS, "and", HB_MAX_EPOCHS)
    print("Eta:", HB_ETA)
    print("Cycles:", HB_CYCLES)
    print("HPO budget:", HB_HPO_EPOCH_UNITS, "epoch units")
    print("Final seeds:", FINAL_EVALUATION_SEEDS)
    print("Device:", device)
    print("=" * 88)

    # Step 1. Build and audit the selected dataset year.
    records, record_statistics = build_clean_records(
        source_root=SOURCE_DATASET_ROOT,
        target_class_set=TARGET_CLASS_SET,
        label_aliases=LABEL_ALIASES,
    )

    full_dataset_audit = audit_records(
        records_to_audit=records,
        class_names=CLASS_NAMES,
    )

    if full_dataset_audit["missing_classes"]:
        raise RuntimeError(
            "Target classes missing from the selected dataset: "
            f"{full_dataset_audit['missing_classes']}"
        )

    save_json(
        data=record_statistics,
        output_json=(
            STATS_ROOT
            / f"{DATASET_TAG}_dataset_scan_summary.json"
        ),
    )
    save_json(
        data=full_dataset_audit,
        output_json=(
            STATS_ROOT
            / f"{DATASET_TAG}_dataset_class_audit.json"
        ),
    )

    print("\nDataset scan:")
    print(json.dumps(record_statistics, indent=2))
    print("\nDataset class audit:")
    print(json.dumps(full_dataset_audit, indent=2))

    plot_class_distribution(
        audit=full_dataset_audit,
        class_names=CLASS_NAMES,
        output_root=RUN_PLOTS_ROOT,
        dataset_tag=DATASET_TAG,
    )

    # Step 2. Prepare the one fixed optimization split.
    optimization_dataset_root = (
        YOLO_DATA_ROOT
        / f"optimization_seed{OPTIMIZATION_SPLIT_SEED}"
    )
    optimization_manifest_root = (
        SPLITS_ROOT
        / DATASET_TAG
        / f"optimization_seed{OPTIMIZATION_SPLIT_SEED}"
    )

    optimization_yaml, optimization_coverage = prepare_yolo_dataset(
        all_records=records,
        output_root=optimization_dataset_root,
        manifest_root=optimization_manifest_root,
        class_names=CLASS_NAMES,
        train_ratio=TRAIN_RATIO,
        valid_ratio=VALID_RATIO,
        test_ratio=TEST_RATIO,
        image_mode=IMAGE_PLACEMENT_MODE,
        split_seed=OPTIMIZATION_SPLIT_SEED,
    )

    if not optimization_coverage:
        raise RuntimeError(
            "Optimization split coverage report is empty."
        )

    # Step 3. Build and verify the complete Hyperband plan.
    hyperband_plan = build_hyperband_plan(
        min_epochs=HB_MIN_EPOCHS,
        max_epochs=HB_MAX_EPOCHS,
        eta=HB_ETA,
        n_cycles=HB_CYCLES,
        random_seed=HB_RANDOM_SEED,
        search_space=SEARCH_SPACE,
        parameter_order=SEARCH_PARAMETER_ORDER,
        expected_unique_configs=EXPECTED_UNIQUE_CONFIGS,
        expected_hpo_evaluations=EXPECTED_HPO_EVALUATIONS,
        expected_full_fidelity_evaluations=(
            EXPECTED_FULL_FIDELITY_EVALUATIONS
        ),
        expected_hpo_epoch_units=HB_HPO_EPOCH_UNITS,
    )

    repeated_plan = build_hyperband_plan(
        min_epochs=HB_MIN_EPOCHS,
        max_epochs=HB_MAX_EPOCHS,
        eta=HB_ETA,
        n_cycles=HB_CYCLES,
        random_seed=HB_RANDOM_SEED,
        search_space=SEARCH_SPACE,
        parameter_order=SEARCH_PARAMETER_ORDER,
        expected_unique_configs=EXPECTED_UNIQUE_CONFIGS,
        expected_hpo_evaluations=EXPECTED_HPO_EVALUATIONS,
        expected_full_fidelity_evaluations=(
            EXPECTED_FULL_FIDELITY_EVALUATIONS
        ),
        expected_hpo_epoch_units=HB_HPO_EPOCH_UNITS,
    )

    if hyperband_plan != repeated_plan:
        raise RuntimeError(
            "Hyperband plan reproducibility check failed."
        )

    save_rows_to_csv_safe(
        rows=flatten_hyperband_plan(hyperband_plan),
        output_csv=plan_csv,
    )

    print("\nHyperband plan:")
    for bracket in hyperband_plan:
        print(
            bracket["bracket_id"],
            "| initial configs:",
            bracket["initial_n"],
            "| budgets:",
            bracket["budgets"],
            "| config IDs:",
            [
                config["config_id"]
                for config in bracket["configs"]
            ],
        )

    # Step 4. Run Hyperband using validation data only.
    hpo_start = time.perf_counter()
    hpo_results: List[Dict[str, Any]] = []
    evaluation_index = 0
    planned_cumulative_epoch_units = 0

    for bracket in hyperband_plan:
        active_configs = list(
            bracket["configs"]
        )

        print("\n" + "#" * 88)
        print(
            "Starting",
            bracket["bracket_id"],
            "with budgets",
            bracket["budgets"],
        )
        print("#" * 88)

        for stage_index, budget_epochs in enumerate(
            bracket["budgets"],
            start=1,
        ):
            warmup_epochs, close_mosaic = get_fidelity_controls(
                budget_epochs=budget_epochs,
                fidelity_controls=FIDELITY_CONTROLS,
            )

            stage_rows: List[Dict[str, Any]] = []

            for config in active_configs:
                evaluation_index += 1
                planned_cumulative_epoch_units += int(
                    budget_epochs
                )
                trial_wall_start = time.perf_counter()

                print(
                    "\nEvaluation",
                    evaluation_index,
                    "|",
                    config["config_id"],
                    "|",
                    budget_epochs,
                    "epochs",
                )

                try:
                    train_result = train_and_validate_configuration(
                        learning_rate=float(
                            config["learning_rate"]
                        ),
                        lrf=float(config["lrf"]),
                        momentum=float(
                            config["momentum"]
                        ),
                        weight_decay=float(
                            config["weight_decay"]
                        ),
                        scale=float(config["scale"]),
                        dataset_yaml=optimization_yaml,
                        project_dir=(
                            HPO_TRIALS_ROOT
                            / bracket["cycle_id"]
                            / bracket["bracket_id"]
                            / (
                                f"stage_{stage_index}_"
                                f"{budget_epochs}ep"
                            )
                        ),
                        run_name=(
                            f"{config['config_id']}_"
                            f"{budget_epochs}ep"
                        ),
                        model_name=MODEL_NAME,
                        epochs=int(budget_epochs),
                        image_size=IMAGE_SIZE,
                        batch_size=BATCH_SIZE,
                        patience=PATIENCE,
                        workers=WORKERS,
                        seed=OPTIMIZATION_TRAIN_SEED,
                        warmup_epochs=warmup_epochs,
                        close_mosaic=close_mosaic,
                        fixed_training_args=FIXED_TRAINING_ARGS,
                        expected_class_names=CLASS_NAMES,
                        strict_fitness_alignment_check=(
                            STRICT_FITNESS_ALIGNMENT_CHECK
                        ),
                        fitness_alignment_tolerance=(
                            FITNESS_ALIGNMENT_TOLERANCE
                        ),
                        device=device,
                    )

                    metrics = train_result["metrics"]

                    result_row = {
                        "status": "PASS",
                        "dataset_tag": DATASET_TAG,
                        "evaluation_index": int(
                            evaluation_index
                        ),
                        "cycle_index": int(
                            bracket["cycle_index"]
                        ),
                        "cycle_id": bracket["cycle_id"],
                        "bracket_id": bracket["bracket_id"],
                        "bracket_number": int(
                            bracket["bracket_number"]
                        ),
                        "stage_index": int(stage_index),
                        "budget_epochs": int(
                            budget_epochs
                        ),
                        "planned_cumulative_epoch_units": int(
                            planned_cumulative_epoch_units
                        ),
                        "config_id": config["config_id"],
                        **{
                            parameter_name: float(
                                config[parameter_name]
                            )
                            for parameter_name
                            in SEARCH_PARAMETER_ORDER
                        },
                        "warmup_epochs": float(
                            warmup_epochs
                        ),
                        "close_mosaic": int(
                            close_mosaic
                        ),
                        "val_precision": float(
                            metrics["precision"]
                        ),
                        "val_recall": float(
                            metrics["recall"]
                        ),
                        "val_map50": float(
                            metrics["map50"]
                        ),
                        "val_map50_95": float(
                            metrics["map50_95"]
                        ),
                        "val_bottom3_mean_map50_95": float(
                            metrics[
                                "bottom3_mean_map50_95"
                            ]
                        ),
                        "fitness_alignment_status": (
                            train_result[
                                "fitness_alignment"
                            ]["status"]
                        ),
                        "training_seconds": float(
                            train_result["timing"][
                                "training_seconds"
                            ]
                        ),
                        "validation_seconds": float(
                            train_result["timing"][
                                "validation_seconds"
                            ]
                        ),
                        "trial_wall_seconds": float(
                            time.perf_counter()
                            - trial_wall_start
                        ),
                        "peak_gpu_memory_gb": (
                            train_result["hardware"][
                                "peak_gpu_memory_gb"
                            ]
                        ),
                        "error_message": "",
                    }

                except Exception as error:
                    result_row = {
                        "status": "FAIL",
                        "dataset_tag": DATASET_TAG,
                        "evaluation_index": int(
                            evaluation_index
                        ),
                        "cycle_index": int(
                            bracket["cycle_index"]
                        ),
                        "cycle_id": bracket["cycle_id"],
                        "bracket_id": bracket["bracket_id"],
                        "bracket_number": int(
                            bracket["bracket_number"]
                        ),
                        "stage_index": int(stage_index),
                        "budget_epochs": int(
                            budget_epochs
                        ),
                        "planned_cumulative_epoch_units": int(
                            planned_cumulative_epoch_units
                        ),
                        "config_id": config["config_id"],
                        **{
                            parameter_name: float(
                                config[parameter_name]
                            )
                            for parameter_name
                            in SEARCH_PARAMETER_ORDER
                        },
                        "warmup_epochs": float(
                            warmup_epochs
                        ),
                        "close_mosaic": int(
                            close_mosaic
                        ),
                        "val_precision": "",
                        "val_recall": "",
                        "val_map50": "",
                        "val_map50_95": "",
                        "val_bottom3_mean_map50_95": "",
                        "fitness_alignment_status": "FAIL",
                        "training_seconds": "",
                        "validation_seconds": "",
                        "trial_wall_seconds": float(
                            time.perf_counter()
                            - trial_wall_start
                        ),
                        "peak_gpu_memory_gb": "",
                        "error_message": privacy_safe_text(
                            value=error,
                            workspace_root=WORKSPACE_ROOT,
                        ),
                    }

                    print("\nHyperband evaluation failed:")
                    print(result_row["error_message"])

                hpo_results.append(result_row)
                stage_rows.append(result_row)

                save_rows_to_csv_safe(
                    rows=hpo_results,
                    output_csv=hpo_results_csv,
                )

            failed_rows = [
                row
                for row in stage_rows
                if row["status"] != "PASS"
            ]

            if failed_rows and REQUIRE_FULL_HPO_BUDGET:
                raise RuntimeError(
                    "A required Hyperband evaluation failed. "
                    "The experiment is stopped rather than "
                    "reporting an under budget comparison."
                )

            if stage_index < len(bracket["budgets"]):
                ranked_stage = rank_hyperband_rows(
                    stage_rows
                )

                if not ranked_stage:
                    raise RuntimeError(
                        "No successful candidates are available "
                        "for Hyperband promotion."
                    )

                keep_count = max(
                    1,
                    int(
                        math.floor(
                            len(ranked_stage) / HB_ETA
                        )
                    ),
                )

                promoted_ids = {
                    str(row["config_id"])
                    for row in ranked_stage[:keep_count]
                }

                print("\nPromotion ranking:")
                for rank, row in enumerate(
                    ranked_stage,
                    start=1,
                ):
                    decision = (
                        "PROMOTE"
                        if str(row["config_id"])
                        in promoted_ids
                        else "STOP"
                    )
                    print(
                        f"{rank}. {row['config_id']} | "
                        f"mAP50-95="
                        f"{float(row['val_map50_95']):.6f} | "
                        f"{decision}"
                    )

                active_configs = [
                    config
                    for config in active_configs
                    if str(config["config_id"])
                    in promoted_ids
                ]

    hpo_seconds = time.perf_counter() - hpo_start

    # Step 5. Audit the HPO budget and freeze the best 40 epoch candidate.
    successful_hpo = [
        row
        for row in hpo_results
        if row["status"] == "PASS"
    ]

    successful_hpo_epoch_units = sum(
        int(row["budget_epochs"])
        for row in successful_hpo
    )

    if REQUIRE_FULL_HPO_BUDGET:
        if (
            len(successful_hpo)
            != EXPECTED_HPO_EVALUATIONS
        ):
            raise RuntimeError(
                "Hyperband completed an unexpected number "
                "of HPO evaluations."
            )

        if (
            successful_hpo_epoch_units
            != HB_HPO_EPOCH_UNITS
        ):
            raise RuntimeError(
                "Hyperband did not complete the full "
                "320 epoch HPO budget."
            )

    full_fidelity_rows = [
        row
        for row in successful_hpo
        if int(row["budget_epochs"])
        == HB_MAX_EPOCHS
    ]

    if (
        REQUIRE_FULL_HPO_BUDGET
        and len(full_fidelity_rows)
        != EXPECTED_FULL_FIDELITY_EVALUATIONS
    ):
        raise RuntimeError(
            "Hyperband completed an unexpected number "
            "of 40 epoch HPO evaluations."
        )

    ranked_full = rank_hyperband_rows(
        full_fidelity_rows
    )

    if not ranked_full:
        raise RuntimeError(
            "No successful 40 epoch Hyperband candidate "
            "is available for final selection."
        )

    for rank, row in enumerate(
        ranked_full,
        start=1,
    ):
        row["full_fidelity_rank"] = int(rank)

    save_rows_to_csv_safe(
        rows=ranked_full,
        output_csv=ranked_full_csv,
    )

    plot_hyperband_progress(
        rows=hpo_results,
        output_root=RUN_PLOTS_ROOT,
        dataset_tag=DATASET_TAG,
    )

    winner = dict(ranked_full[0])

    selected_config = {
        "method": "Hyperband 20 and 40 epoch fidelities, two cycles",
        "fitness": "validation mAP50-95",
        "test_set_used_for_selection": False,
        "selected_config_id": winner["config_id"],
        "selected_cycle_id": winner["cycle_id"],
        "selected_bracket_id": winner["bracket_id"],
        "selected_budget_epochs": int(
            winner["budget_epochs"]
        ),
        **{
            parameter_name: float(
                winner[parameter_name]
            )
            for parameter_name
            in SEARCH_PARAMETER_ORDER
        },
        "selected_val_precision": float(
            winner["val_precision"]
        ),
        "selected_val_recall": float(
            winner["val_recall"]
        ),
        "selected_val_map50": float(
            winner["val_map50"]
        ),
        "selected_val_map50_95": float(
            winner["val_map50_95"]
        ),
        "hpo_epoch_units": int(
            HB_HPO_EPOCH_UNITS
        ),
        "successful_hpo_epoch_units": int(
            successful_hpo_epoch_units
        ),
        "hyperband_sampling_seed": int(
            HB_RANDOM_SEED
        ),
        "optimization_split_seed": int(
            OPTIMIZATION_SPLIT_SEED
        ),
        "optimization_train_seed": int(
            OPTIMIZATION_TRAIN_SEED
        ),
    }

    save_json(
        data=selected_config,
        output_json=selected_json,
    )

    save_rows_to_csv_safe(
        rows=boundary_proximity_report(
            selected_config=selected_config,
            search_space=SEARCH_SPACE,
            parameter_order=SEARCH_PARAMETER_ORDER,
        ),
        output_csv=boundary_csv,
    )

    print("\nSelected Hyperband configuration:")
    print(json.dumps(selected_config, indent=2))

    # Step 6. Retrain the frozen configuration on three split and train seeds.
    final_start = time.perf_counter()
    final_rows: List[Dict[str, Any]] = []
    per_class_rows: List[Dict[str, Any]] = []

    for seed in FINAL_EVALUATION_SEEDS:
        final_wall_start = time.perf_counter()

        print("\n" + "#" * 88)
        print(
            "Final split and train seed:",
            seed,
        )
        print("#" * 88)

        try:
            final_dataset_root = (
                YOLO_DATA_ROOT
                / f"final_seed{seed}"
            )
            final_manifest_root = (
                SPLITS_ROOT
                / DATASET_TAG
                / f"final_seed{seed}"
            )

            final_yaml, final_coverage = prepare_yolo_dataset(
                all_records=records,
                output_root=final_dataset_root,
                manifest_root=final_manifest_root,
                class_names=CLASS_NAMES,
                train_ratio=TRAIN_RATIO,
                valid_ratio=VALID_RATIO,
                test_ratio=TEST_RATIO,
                image_mode=IMAGE_PLACEMENT_MODE,
                split_seed=seed,
            )

            if not final_coverage:
                raise RuntimeError(
                    "Final split coverage report is empty."
                )

            final_train_result = train_and_validate_configuration(
                learning_rate=float(
                    selected_config["learning_rate"]
                ),
                lrf=float(
                    selected_config["lrf"]
                ),
                momentum=float(
                    selected_config["momentum"]
                ),
                weight_decay=float(
                    selected_config["weight_decay"]
                ),
                scale=float(
                    selected_config["scale"]
                ),
                dataset_yaml=final_yaml,
                project_dir=(
                    FINAL_TRAINING_ROOT
                    / f"seed_{seed}"
                ),
                run_name=f"final_seed_{seed}",
                model_name=MODEL_NAME,
                epochs=FINAL_EPOCHS,
                image_size=IMAGE_SIZE,
                batch_size=BATCH_SIZE,
                patience=PATIENCE,
                workers=WORKERS,
                seed=seed,
                warmup_epochs=FINAL_WARMUP_EPOCHS,
                close_mosaic=FINAL_CLOSE_MOSAIC,
                fixed_training_args=FIXED_TRAINING_ARGS,
                expected_class_names=CLASS_NAMES,
                strict_fitness_alignment_check=(
                    STRICT_FITNESS_ALIGNMENT_CHECK
                ),
                fitness_alignment_tolerance=(
                    FITNESS_ALIGNMENT_TOLERANCE
                ),
                device=device,
            )

            test_result = evaluate_checkpoint_on_test_split(
                checkpoint_path=(
                    final_train_result["paths"][
                        "best_checkpoint"
                    ]
                ),
                dataset_yaml=final_yaml,
                output_root=(
                    TEST_EVAL_ROOT
                    / f"seed_{seed}"
                ),
                run_name=f"test_seed_{seed}",
                expected_class_names=CLASS_NAMES,
                image_size=IMAGE_SIZE,
                batch_size=BATCH_SIZE,
                workers=WORKERS,
                device=device,
            )

            final_row = {
                "status": "PASS",
                "dataset_tag": DATASET_TAG,
                "split_train_seed": int(seed),
                "selected_config_id": (
                    selected_config[
                        "selected_config_id"
                    ]
                ),
                **{
                    parameter_name: (
                        selected_config[
                            parameter_name
                        ]
                    )
                    for parameter_name
                    in SEARCH_PARAMETER_ORDER
                },
                "final_val_precision": float(
                    final_train_result["metrics"][
                        "precision"
                    ]
                ),
                "final_val_recall": float(
                    final_train_result["metrics"][
                        "recall"
                    ]
                ),
                "final_val_map50": float(
                    final_train_result["metrics"][
                        "map50"
                    ]
                ),
                "final_val_map50_95": float(
                    final_train_result["metrics"][
                        "map50_95"
                    ]
                ),
                "test_precision": float(
                    test_result["test_precision"]
                ),
                "test_recall": float(
                    test_result["test_recall"]
                ),
                "test_map50": float(
                    test_result["test_map50"]
                ),
                "test_map50_95": float(
                    test_result["test_map50_95"]
                ),
                "test_bottom3_mean_map50_95": float(
                    test_result[
                        "test_bottom3_mean_map50_95"
                    ]
                ),
                "fitness_alignment_status": (
                    final_train_result[
                        "fitness_alignment"
                    ]["status"]
                ),
                "final_training_seconds": float(
                    final_train_result["timing"][
                        "training_seconds"
                    ]
                ),
                "final_validation_seconds": float(
                    final_train_result["timing"][
                        "validation_seconds"
                    ]
                ),
                "test_eval_seconds": float(
                    test_result["test_seconds"]
                ),
                "final_run_wall_seconds": float(
                    time.perf_counter()
                    - final_wall_start
                ),
                "error_message": "",
            }

            for class_row in test_result["per_class_rows"]:
                per_class_rows.append({
                    "dataset_tag": DATASET_TAG,
                    "split_train_seed": int(seed),
                    **class_row,
                })

        except Exception as error:
            final_row = {
                "status": "FAIL",
                "dataset_tag": DATASET_TAG,
                "split_train_seed": int(seed),
                "selected_config_id": (
                    selected_config[
                        "selected_config_id"
                    ]
                ),
                **{
                    parameter_name: (
                        selected_config[
                            parameter_name
                        ]
                    )
                    for parameter_name
                    in SEARCH_PARAMETER_ORDER
                },
                "final_val_precision": "",
                "final_val_recall": "",
                "final_val_map50": "",
                "final_val_map50_95": "",
                "test_precision": "",
                "test_recall": "",
                "test_map50": "",
                "test_map50_95": "",
                "test_bottom3_mean_map50_95": "",
                "fitness_alignment_status": "FAIL",
                "final_training_seconds": "",
                "final_validation_seconds": "",
                "test_eval_seconds": "",
                "final_run_wall_seconds": float(
                    time.perf_counter()
                    - final_wall_start
                ),
                "error_message": privacy_safe_text(
                    value=error,
                    workspace_root=WORKSPACE_ROOT,
                ),
            }

            print("\nFinal evaluation failed:")
            print(final_row["error_message"])

        final_rows.append(final_row)

        save_rows_to_csv_safe(
            rows=final_rows,
            output_csv=final_csv,
        )
        save_rows_to_csv_safe(
            rows=per_class_rows,
            output_csv=per_class_csv,
        )

    final_seconds = time.perf_counter() - final_start

    successful_final = [
        row
        for row in final_rows
        if row["status"] == "PASS"
    ]

    if (
        REQUIRE_ALL_FINAL_RUNS
        and len(successful_final)
        != len(FINAL_EVALUATION_SEEDS)
    ):
        raise RuntimeError(
            "All final seed runs are required. "
            f"Completed {len(successful_final)} of "
            f"{len(FINAL_EVALUATION_SEEDS)}."
        )

    # Step 7. Save final summaries and experiment manifest.
    summary_metric_names = [
        "final_val_precision",
        "final_val_recall",
        "final_val_map50",
        "final_val_map50_95",
        "test_precision",
        "test_recall",
        "test_map50",
        "test_map50_95",
        "test_bottom3_mean_map50_95",
        "final_training_seconds",
        "final_validation_seconds",
        "test_eval_seconds",
        "final_run_wall_seconds",
    ]

    summary_rows = summarize_metric_mean_sd(
        rows=successful_final,
        metric_names=summary_metric_names,
    )

    def append_single_value_summary(
        metric_name: str,
        metric_value: float,
    ) -> None:
        summary_rows.append({
            "metric": metric_name,
            "n": 1,
            "mean": float(metric_value),
            "sd": "",
            "min": float(metric_value),
            "max": float(metric_value),
        })

    total_seconds = (
        time.perf_counter()
        - workflow_start
    )

    append_single_value_summary(
        "hpo_epoch_units",
        HB_HPO_EPOCH_UNITS,
    )
    append_single_value_summary(
        "successful_hpo_epoch_units",
        successful_hpo_epoch_units,
    )
    append_single_value_summary(
        "hpo_successful_evaluations",
        len(successful_hpo),
    )
    append_single_value_summary(
        "hpo_full_fidelity_evaluations",
        len(full_fidelity_rows),
    )
    append_single_value_summary(
        "final_evaluation_epoch_units",
        FINAL_EVALUATION_EPOCH_UNITS,
    )
    append_single_value_summary(
        "hpo_stage_seconds",
        hpo_seconds,
    )
    append_single_value_summary(
        "final_stage_seconds",
        final_seconds,
    )
    append_single_value_summary(
        "total_workflow_seconds",
        total_seconds,
    )

    per_class_summary = summarize_per_class_mean_sd(
        per_class_rows
    )

    save_rows_to_csv_safe(
        rows=summary_rows,
        output_csv=final_summary_csv,
    )
    save_rows_to_csv_safe(
        rows=per_class_summary,
        output_csv=per_class_summary_csv,
    )

    plot_final_test_metrics(
        final_rows=final_rows,
        output_root=RUN_PLOTS_ROOT,
        dataset_tag=DATASET_TAG,
    )

    manifest = {
        "dataset_tag": DATASET_TAG,
        "method": "Hyperband",
        "model_name": MODEL_NAME,
        "fitness": "validation mAP50-95",
        "test_set_used_during_hpo": False,
        "search_space_version": SEARCH_SPACE_VERSION,
        "search_space": SEARCH_SPACE,
        "search_parameter_order": SEARCH_PARAMETER_ORDER,
        "fixed_training_args": FIXED_TRAINING_ARGS,
        "runtime": {
            "image_size": IMAGE_SIZE,
            "batch_size": BATCH_SIZE,
            "workers": WORKERS,
            "patience": PATIENCE,
            "cache": FIXED_TRAINING_ARGS["cache"],
            "deterministic": True,
        },
        "hyperband": {
            "min_epochs": HB_MIN_EPOCHS,
            "max_epochs": HB_MAX_EPOCHS,
            "eta": HB_ETA,
            "cycles": HB_CYCLES,
            "sampling_seed": HB_RANDOM_SEED,
            "fidelity_controls": FIDELITY_CONTROLS,
            "promoted_checkpoint_resume": False,
            "planned_hpo_epoch_units": (
                HB_HPO_EPOCH_UNITS
            ),
            "successful_hpo_epoch_units": (
                successful_hpo_epoch_units
            ),
            "expected_unique_configs": (
                EXPECTED_UNIQUE_CONFIGS
            ),
            "expected_hpo_evaluations": (
                EXPECTED_HPO_EVALUATIONS
            ),
            "expected_full_fidelity_evaluations": (
                EXPECTED_FULL_FIDELITY_EVALUATIONS
            ),
        },
        "seeds": {
            "optimization_split_seed": (
                OPTIMIZATION_SPLIT_SEED
            ),
            "optimization_train_seed": (
                OPTIMIZATION_TRAIN_SEED
            ),
            "final_evaluation_seeds": (
                FINAL_EVALUATION_SEEDS
            ),
        },
        "selected_config": selected_config,
        "software": {
            "python": sys.version.split()[0],
            "ultralytics": ultralytics.__version__,
            "torch": torch.__version__,
            "cuda_available": (
                torch.cuda.is_available()
            ),
            "gpu_name": (
                torch.cuda.get_device_name(0)
                if torch.cuda.is_available()
                else None
            ),
        },
        "timing_seconds": {
            "hpo_stage": float(hpo_seconds),
            "final_stage": float(final_seconds),
            "total_workflow": float(total_seconds),
        },
        "output_root": relative_output_path(
            path=RUN_ROOT,
            workspace_root=WORKSPACE_ROOT,
        ),
    }

    save_json(
        data=manifest,
        output_json=manifest_json,
    )

    print("\nFinal mean and sample SD:")
    for row in summary_rows:
        if row["mean"] == "":
            continue

        if row["sd"] == "":
            print(
                f"{row['metric']}: "
                f"{float(row['mean']):.4f}"
            )
        else:
            print(
                f"{row['metric']}: "
                f"{float(row['mean']):.4f} "
                f"+/- {float(row['sd']):.4f}"
            )

    print("\nSaved result tables:")
    for saved_path in [
        plan_csv,
        hpo_results_csv,
        ranked_full_csv,
        selected_json,
        boundary_csv,
        final_csv,
        final_summary_csv,
        per_class_csv,
        per_class_summary_csv,
        manifest_json,
    ]:
        print(
            relative_output_path(
                path=saved_path,
                workspace_root=WORKSPACE_ROOT,
            )
        )

    # Step 8. Remove large temporary folders only after every required run succeeds.
    if YOLO_DATA_ROOT.exists():
        shutil.rmtree(YOLO_DATA_ROOT)

    if HPO_TRIALS_ROOT.exists():
        shutil.rmtree(HPO_TRIALS_ROOT)

    print("\nCleanup completed:")
    print(
        "Removed:",
        relative_output_path(
            path=YOLO_DATA_ROOT,
            workspace_root=WORKSPACE_ROOT,
        ),
    )
    print(
        "Removed:",
        relative_output_path(
            path=HPO_TRIALS_ROOT,
            workspace_root=WORKSPACE_ROOT,
        ),
    )

    return {
        "selected_config": selected_config,
        "hpo_results": hpo_results,
        "ranked_full_fidelity": ranked_full,
        "final_rows": final_rows,
        "summary_rows": summary_rows,
        "per_class_summary_rows": (
            per_class_summary
        ),
        "hpo_stage_seconds": float(
            hpo_seconds
        ),
        "final_stage_seconds": float(
            final_seconds
        ),
        "total_workflow_seconds": float(
            total_seconds
        ),
        "output_root": relative_output_path(
            path=RUN_ROOT,
            workspace_root=WORKSPACE_ROOT,
        ),
    }


## Module 10: Run switch

Leave the switch `False` while reviewing the notebook. Set it to `True` only when the dataset is in place and you are ready to run the complete 320 epoch Hyperband search followed by the three final 40 epoch evaluations.


In [ ]:
RUN_HYPERBAND = False

if RUN_HYPERBAND:
    hyperband_result = run_hyperband_experiment()
else:
    print("RUN_HYPERBAND is False.")
    print("Dataset:", DATASET_TAG)
    print("HPO budget:", HB_HPO_EPOCH_UNITS, "epoch units")
    print(
        "Final evaluation budget:",
        FINAL_EVALUATION_EPOCH_UNITS,
        "epoch units",
    )
    print("Test data is not used during HPO.")
